# Tiered Spherical Parameterization - fail-early, Tier-0 first

Arbitrary genus-0 mesh -> SHP parameterization, built as a coarse-to-fine
pyramid on a **patch** foundation. A *global* Brechbuhler map is not bijective
for shapes with thin features (e.g. the mushroom stalk); the robust path is to
segment into well-behaved disk-like patches, map a coarse **cage** to the sphere,
then fill patch interiors by Laplace solves.

**Fail-early philosophy.** There is no point continuing to finer tiers if even a
single triangle is flipped at tier 0. So we make Tier 0 bullet-proof first:

1. Curvature-adaptive dense high-quality mesh (~3k verts) - thin structures get
   high triangle counts, so the cage is well-behaved.
2. Tier-0 patches: a small, user-specified number (3-30), with **no annular and
   no cap patches**.
3. Rescue: if caps / annuli remain across the whole range, **subdivide** each
   problematic patch into `n_split` sub-patches and re-check.
4. Bijective cage map with **area optimization + shear constraint** (flip-prevented).
5. Strict gate: no flipped / intersecting faces, no highly sheared thin faces.
6. Low-order SHP shape (`L_max=3`) of the tier-0 parameterization.
7. Only if all of the above pass, proceed to the next tier.

## Imports and setup

In [ ]:
import numpy as np
import sys, os, importlib
import matplotlib.pyplot as plt
import pyvista as pv
pv.set_jupyter_backend('static')

code_dir = r'C:\Users\Khaled Khairy\Dropbox\Projects\hot\Project_spherical_parameterization\code'
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

from pySHP.surface_mesh import surface_mesh
from pySHP.shp_surface import shp_surface
from pySHP.utils import readoff, kk_sph2cart

import pySHP.level2.tiered_spherical_parameterization as tsp
importlib.reload(tsp)
print('tiered module:', tsp.__file__)

## Helper: display a tier (sphere parameterization + SHP reconstruction)

In [ ]:
def show_tier(tier, nico=4):
    m = tier.mesh
    u, v, w = kk_sph2cart(m.t, m.p, np.ones(len(m.t)))
    Xs = np.column_stack([u, v, w])
    cells = np.hstack((np.full((len(m.F), 1), 3), m.F)).astype(np.int64).flatten()
    sph = pv.PolyData(Xs, cells)
    pl = pv.Plotter(shape=(1, 2), window_size=(1100, 520))
    pl.subplot(0, 0)
    labels = getattr(m, 'face_labels', None)
    if labels is not None:
        sph.cell_data['patch'] = np.asarray(labels)
        pl.add_mesh(sph, scalars='patch', show_edges=True, edge_color='black',
                    line_width=0.4, cmap='tab20')
    else:
        pl.add_mesh(sph, color='lightblue', show_edges=True, edge_color='black',
                    line_width=0.4)
    d = tier.diagnostics
    pl.add_text(f"L{tier.level} {tier.kind}: sphere param\n"
                f"folds={d.get('n_foldovers','?')}  "
                f"max_shear={d.get('tier0_gate',{}).get('max_shear',float('nan')):.2f}  "
                f"mean_q={d.get('mean_quality',float('nan')):.2f}", font_size=9)
    pl.background_color = 'white'
    pl.subplot(0, 1)
    if tier.shp is not None:
        m_shp, X_shp, F_shp, *_ = tier.shp.get_mesh(nico=nico)
        cshp = np.hstack((np.full((len(F_shp), 1), 3), F_shp)).astype(np.int64).flatten()
        pl.add_mesh(pv.PolyData(X_shp, cshp), color='lightgreen', show_edges=True,
                    edge_color='black', line_width=0.3)
        pl.add_text(f"SHP L_max={tier.shp.L_max} (RMS={tier.shp_rms:.4f})", font_size=9)
    pl.background_color = 'white'
    pl.show()
print('show_tier ready')

## Load mesh

In [ ]:
topic = 'test_set'
file_name = 'mushroom_repaired_03.off'
fn_shape = os.path.join(code_dir, 'Matlab', 'shp_toolbox-main', 'shp_toolbox-main',
                        'test_data', 'off', topic, file_name)
X_orig, F_orig = readoff(fn_shape)
m_orig = surface_mesh(X_orig, F_orig)
print(f'Input: {len(m_orig.X)} verts, {len(m_orig.F)} faces')
m_orig.plot()

## [1] Curvature-adaptive dense high-quality mesh (~3k verts)

Thin structures (the stalk) get many triangles here, which is exactly what makes
the tier-0 cage well-behaved. Use `target_verts=3000` for production; lower for
fast iteration.

In [ ]:
m_fine, q_stats = tsp.preprocess_mesh(
    m_orig, target_verts=3000, curvature_strength=2.0, verbose=True)
print('quality:', {k: round(v, 3) for k, v in q_stats.items()
                   if isinstance(v, (int, float))})
m_fine.plot_H()

## [2]+[3] Tier-0 patches: no cap / no annular (rescue by subdivision)

`generate_tier0_patches` sweeps the user range, and for any cap (< `min_neighbors`
neighbours) or annular (>1 boundary loop) patch it subdivides into `n_split`
sub-patches and re-checks. It raises (fail early) if it cannot produce a clean
labelling - so a green result here means every patch is a regular disk.

In [ ]:
labels, seg_meta = tsp.generate_tier0_patches(
    m_fine, nseeds_range=(3, 30), min_neighbors=3, n_split=5,
    max_rescue_rounds=2, verbose=True)
print(f"\nAccepted: nseeds={seg_meta['nseeds']}, rescued={seg_meta['rescued']}, "
      f"{seg_meta['n_patches']} patches")

# Classification table (proof: no caps, no annuli)
info, problematic = tsp.classify_patches(m_fine, labels, min_neighbors=3)
print(f"{'patch':>5} {'neighbors':>9} {'bnd_loops':>9} {'cap':>5} {'annular':>7}")
for lab in sorted(info):
    r = info[lab]
    print(f"{lab:>5} {r['neighbors']:>9} {r['boundary_components']:>9} "
          f"{str(r['is_cap']):>5} {str(r['is_annular']):>7}")
assert not problematic, f"problematic patches remain: {problematic}"

m_fine.face_labels = labels
m_fine.plot_labels()

## Build the decimated cage from the accepted labels

In [ ]:
m_seg, PM = tsp.build_cage_from_labels(
    m_fine, labels, target_ratio=0.25, curvature_weight=1.0, verbose=True)
print(f"cage: {len(PM['pm'].X)} verts, {len(PM['pm'].F)} faces, "
      f"patches={PM['npatches']}")

from pySHP.level1.diagnose_simplified_mesh import diagnose_simplified_mesh_full
cage_diag = diagnose_simplified_mesh_full(PM, verbose=True)
print('cage topology valid:', cage_diag['valid'])

## [4] Tier-0 bijective map: area optimization + shear constraint (flip-prevented)

In [ ]:
cage = tsp.parameterize_cage(PM, area_niter=40, area_step=0.1,
                            shear_niter=40, shear_step=0.2, verbose=True)
tier0 = tsp.Tier(level=0, mesh=cage, kind='cage')
tsp.compute_tier_diagnostics(tier0, verbose=True)

## [5] Strict tier-0 success gate

Pass requires: zero flipped/intersecting faces, area closed (sum ~ 4*pi), shear
below threshold, and no thin faces. Tune `max_shear` / `min_quality` to taste.

In [ ]:
tier0_ok, tier0_gate = tsp.validate_tier0(
    tier0, max_shear=2.0, min_quality=0.05, verbose=True)
print('\nTIER-0 SUCCESS:', tier0_ok)

## [6] Low-order SHP shape (L_max=3) of the tier-0 parameterization

In [ ]:
tsp.fit_tier_shp(tier0, L_max=3, verbose=True)
show_tier(tier0)

## [7] Gate before proceeding

Only continue to the full patch fill + finer tiers if tier 0 passed. Otherwise,
iterate on steps [2]-[4] (range, `n_split`, optimization budget) until tier 0 is
clean.

In [ ]:
if not tier0_ok:
    raise RuntimeError(
        'Tier 0 did not pass the gate. Fix tier 0 before proceeding '
        '(widen nseeds range, raise n_split, or increase area/shear iterations).')
print('Tier 0 passed - proceeding to finer tiers.')

## Stage 3 - Full patch fill (authoritative parameterization)

In [ ]:
m_full = tsp.fill_full_patches(PM, m_seg, verbose=True)
m_full.plot_spherical_parameterization(flag=0, face_field=m_full.face_labels)

## Stage 4/5 - Interior tiers (decimate boundary-protected + re-fill + polish)

In [ ]:
tiers_interior = tsp.build_interior_tiers(
    m_full, n_interior_tiers=3, coarsest_faces=600,
    area_niter=50, shear_niter=30, verbose=True)
for t in tiers_interior:
    tsp.compute_tier_diagnostics(t, verbose=True)
    tsp.fit_tier_shp(t, L_max=12, verbose=True)
tiers = [tier0] + tiers_interior
print('\nTiers:', [str(t) for t in tiers])

### Display every tier (sphere parameterization + SHP shape), coarse -> fine

In [ ]:
for t in tiers:
    print(f"--- Tier {t.level} ({t.kind}) ---")
    show_tier(t)

## Summary table (coarse -> fine)

In [ ]:
print(f"{'tier':>4} {'kind':>9} {'verts':>6} {'faces':>6} "
      f"{'bij':>4} {'folds':>6} {'areaXS%':>8} {'corr':>6} {'meanQ':>6} {'shpRMS':>8}")
for t in tiers:
    d = t.diagnostics
    print(f"{t.level:>4} {t.kind:>9} {t.n_verts:>6} {t.n_faces:>6} "
          f"{str(d.get('bijective')):>4} {d.get('n_foldovers',-1):>6} "
          f"{100*d.get('area_excess_rel',float('nan')):>8.1f} "
          f"{d.get('area_correlation',float('nan')):>6.2f} "
          f"{d.get('mean_quality',float('nan')):>6.2f} "
          f"{t.shp_rms if t.shp_rms is not None else float('nan'):>8.4f}")

## One-shot convenience (fail-early end-to-end)

`run_tiered_pipeline` runs all of the above; it stops and returns only tier 0 if
the tier-0 gate fails (`stop_on_tier0_failure=True`).

In [ ]:
# tiers_all, art = tsp.run_tiered_pipeline(
#     m_orig, target_verts=3000, n_tiers=4, nseeds_range=(3, 30),
#     n_split=5, max_rescue_rounds=2, coarsest_faces=600,
#     cage_area_niter=40, cage_shear_niter=40,
#     tier0_max_shear=2.0, tier0_min_quality=0.05,
#     stop_on_tier0_failure=True, L_max=16, tier0_L_max=3, verbose=True)
# for t in tiers_all:
#     show_tier(t)

## Export finest-tier SHP

In [ ]:
out_shp3 = os.path.join(code_dir, 'pySHP', 'tests', 'tiered_parameterized_mesh.shp3')
tiers[-1].shp.export_shp3(out_shp3)
print('Exported', out_shp3, '| reconstruction RMS =', tiers[-1].shp_rms)